In [5]:
# -*- coding: utf-8 -*-
"""
A demonstration benchmarks the MCU-Quake (5-20) using 100 samples from the UUSS dataset. The full data are provided in the Data Availability section of the manuscript. 

Note that the performance of MCU-Quake can be improved by employing a more sophisticated interpretation method for the model's output embeddings.
"""


from Library import utils, dataset
import pandas as pd
import os
from tensorflow import keras
from tqdm import tqdm
import numpy as np
from datetime import datetime
import logging
import config

def plot_save_mismatches(plot_func, mismatch_pred, name_mismatch_pred,
                         source_data, source_meta,
                         plot_num, save_dir,
                         input_win=None):
        # save fig dir
    if len(mismatch_pred)>1:
        _plot_num = min(plot_num, len(mismatch_pred))
        _fig_dir = os.path.join(save_dir, f"Figures, {name_mismatch_pred.split('.')[0]}")
        if not os.path.exists(_fig_dir): os.makedirs(_fig_dir)
        _id_list = np.random.choice(list(mismatch_pred.keys()), size=_plot_num, replace=False)
        plot_func(_id_list, source_data, source_meta,
                  pred_info=mismatch_pred, save_dir=_fig_dir,
                  input_win=input_win)




if __name__ == "__main__":
                    
    #===================================================================
    #                    3-component dataset and embeddings
    #===================================================================

    source_to_code = {"noise": 0, "se": 1}
    code_to_source = {0: "noise", 1: "se"}

    # --- Karena menggunakan mac GANTI BAGIAN INI ---
    # key_data_dir = r"Data benchmark-demo"
    # model_path = r"Pre-trained model\MCU-Quake 5-20"
    # embedding_3C_dir = r"Typical embedding\Embedding_data train 3C, UUSS n11275 std15, 30120909"

    # --- BAGIAN KONFIGURASI PATH ---
    base_path = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/rep_code"

    # 1. Tentukan folder data
    key_data_dir = '/Volumes/Extreme SSD/stream_stead/data_stead'

    # 2. DEFINISIKAN NAMA FILE (Ini yang tadi hilang/error)
    key_test_file = "STEAD_MCQUAKE_5000_unseen.json"

    # 3. Selebihnya...
    DATA_TAG = "UUSS"
    true_labels = ["NO", "SE"]
    model_path = os.path.join(base_path, "Pre-trained model", "MCU-Quake 5-20")
    embedding_3C_dir = os.path.join(base_path, "Typical embedding", "Embedding_data train 3C, UUSS n11275 std15, 30120909")


    """ STEAD dataset """
    # DATA_TAG = "STEAD" 

    # key_test_file = "STEAD MCU, test n15275 r100.json"
    # true_labels = ["NO", "LE"]


    # ''' embeddings, STEAD dataset '''
    # embedding_3C_dir = r"Typical embedding\Embedding_data train 3C, STEAD norm7 mag3 L n61099, 30172538"


    #===================================================================
    #                        MCU-Quake: 5-20, 7-second
    #===================================================================

    """ model config """

    # 7s model
    INPUT_WIN = 7  # seconds
    SAMPLING_RATE = 100

    MODEL_TAG = "MCU_5-20"

    # model
    model_path = r"Pre-trained model\MCU-Quake 5-20"



    #===================================================================
    #                           Prepare files
    #===================================================================

    """ Load dataset """
    """ Load dataset """
    print(f"[INFO] load test dataset ...")
    test_data = dataset.load_json_data(os.path.join(key_data_dir, key_test_file)) 

    NUM_RECORD = len(test_data)   # all the dataset

    """ create log file """
    # KOREKSI: Gunakan path SSD Extreme secara langsung
    SAVE_BASE = "/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo"

    # Pastikan folder utama dibuat jika belum ada
    if not os.path.exists(SAVE_BASE):
        os.makedirs(SAVE_BASE)

    now = datetime.now()
    time_str = now.strftime("%d%H%M%S")
    
    # Gunakan underscore (_) sebagai pengganti koma atau spasi agar aman di sistem file
    folder_name = f"{MODEL_TAG}_{DATA_TAG}_{time_str}"
    save_dir = os.path.join(SAVE_BASE, folder_name)
    
    if not os.path.exists(save_dir): 
        os.makedirs(save_dir)

    # Konfigurasi Logger (Simpan Log ke SSD)
    log_file_path = os.path.join(save_dir, "task_log.txt")
    logging.basicConfig(filename=log_file_path,
                        level=logging.INFO,
                        format='%(asctime)s - [%(levelname)s]: %(message)s',
                        datefmt = "%Y-%m-%d %H:%M:%S",
                        filemode='w')
    
    logger = logging.getLogger()
    # Hapus handler lama jika ada agar tidak double logging
    if logger.hasHandlers():
        logger.handlers.clear()
        
    logger.addHandler(logging.StreamHandler())

    logger.info(f"Task summary:\n"
                f"Dataset: {os.path.join(key_data_dir, key_test_file)}\n"
                f"Model: {MODEL_TAG}, key embeddings dir: {embedding_3C_dir}\n"
                f"True labels: {true_labels}\n"
                f"Source to code: {source_to_code}\n"
                f"Code to source: {code_to_source}\n"
                f"Output Directory: {save_dir}\n"
                )

    logger.info(f"Load data files ...")

    # embedding model
    # --- 1. Load Embedding Model ---
    # Definisikan path model secara absolut
    final_model_path = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20'
    
    # Load model menggunakan path tersebut
    embedding_model = keras.models.load_model(filepath=final_model_path)

    # --- 2. Load Embedding JSON Data ---
    # Definisikan path folder embedding secara absolut
    final_emb_dir = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909'

    train_embedding_Z_file = "Embedding data, Z.json"
    train_embedding_N_file = "Embedding data, N.json"
    train_embedding_E_file = "Embedding data, E.json"

    # Gunakan final_emb_dir yang sudah kita buat
    embedding_Z = dataset.load_embedding_data(final_emb_dir, train_embedding_Z_file)
    embedding_N = dataset.load_embedding_data(final_emb_dir, train_embedding_N_file)
    embedding_E = dataset.load_embedding_data(final_emb_dir, train_embedding_E_file)

    # --- 3. Hitung Statistik ---
    logger.info(f"Estimate embeddings statistics ...")
    embeddings_Z_PDFs = utils.embedding_PDFs_1D(embedding_Z)
    embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)


    #===================================================================
#                            Evaluation (Z-only, 1C)
#===================================================================

    # 1C: noise vs se (seismic)
    total_true_1C_KDE = []   # fill source_code
    total_pred_1C_KDE = []

    total_true_1C_Norm = []  # fill source_code
    total_pred_1C_Norm = []

    # mismatch result (1C only)
    # {"ID":
    #    {"true": int,
    #     "pred": int,
    #     "softmax[NO,SE]": list,
    #    }
    # }
    mismatch_noise_1C_KDE   = {}
    mismatch_seismic_1C_KDE = {}


    """ extract embedding then make inference """

    logger.info(f"Evaluate metrics on test dataset ...")

    num_points = int(INPUT_WIN*SAMPLING_RATE)

    keys_list = list(test_data.keys())[:NUM_RECORD]

    # mmulai baris ini diganti 1c
     
    for i in tqdm(range(len(keys_list)), desc="Progress", position=0):

        # Normalisasi label menjadi 2 kelas
        raw_label = record["type"].lower()

        if raw_label in ["qb", "le"]:
            quake_label = "se"
        else:
            quake_label = "noise"


        #quake_label = record["type"].lower()

        # ============================
        # 1. LOAD Z-ONLY DATA
        # ============================
        Z_noise_data = record["Z_noise"][-num_points:]
        Z_data       = record["Z"][:num_points]

        # ============================
        # 2. EMBEDDING (Z ONLY)
        # ============================
        _input_Z_noise = utils.latent_codes_1D(Z_noise_data, embedding_model)
        _input_Z       = utils.latent_codes_1D(Z_data, embedding_model)

        # ============================
        # 3. INFERENCE 1C (KDE)
        # ============================
        _noise_or_quake, _noise_likelihood, _noise_softmax = \
            utils.infer_1C_PDFs(_input_Z_noise, embeddings_Z_PDFs, choose_pdf="Kernel")

        _sources_code, _source_likelihood, _source_softmax = \
            utils.infer_1C_PDFs(_input_Z, embeddings_Z_PDFs, choose_pdf="Kernel")

        # noise
        total_true_1C_KDE.append(source_to_code["noise"])
        total_pred_1C_KDE.append(_noise_or_quake)

        if source_to_code["noise"] != _noise_or_quake:
            mismatch_noise_1C_KDE[record_key] = {
                "true": source_to_code["noise"],
                "pred": int(_noise_or_quake),
                "softmax[NO,SE]": _noise_softmax
            }

        # seismic
        total_true_1C_KDE.append(source_to_code[quake_label])
        total_pred_1C_KDE.append(_sources_code)

        if source_to_code[quake_label] != _sources_code:
            mismatch_seismic_1C_KDE[record_key] = {
                "true": source_to_code[quake_label],
                "pred": int(_sources_code),
                "softmax[NO,SE]": _source_softmax
            }
    
        # Normalisasi prediksi ke 2 kelas
        if _noise_or_quake in [1, 2]:
            _noise_or_quake = 1
        else:
            _noise_or_quake = 0

        if _sources_code in [1, 2]:
            _sources_code = 1
        else:
            _sources_code = 0
            

        # ============================
        # 4. INFERENCE 1C (Normal)
        # ============================
        _noise_or_quake, _, _ = utils.infer_1C_PDFs(
            _input_Z_noise, embeddings_Z_PDFs, choose_pdf="Norm"
        )
        _sources_code, _, _ = utils.infer_1C_PDFs(
            _input_Z, embeddings_Z_PDFs, choose_pdf="Norm"
        )

        total_true_1C_Norm.append(source_to_code["noise"])
        total_pred_1C_Norm.append(_noise_or_quake)

        total_true_1C_Norm.append(source_to_code[quake_label])
        total_pred_1C_Norm.append(_sources_code)

        # ============================
        # 5. SKIP 3C BENCHMARKING
        # ============================
        pass



    """ calculate metrics and plot """

    ''' 1C confusion metrics '''

    # 1C confusion matrix, KDE
    matrix_1C_KDE, metrics_1C_KDE = utils.calc_confusion_metrics(total_true_1C_KDE, total_pred_1C_KDE)

    title_1C_KDE = f"{MODEL_TAG} {DATA_TAG}, 1C, KDE"
    fig_1C_KDE = utils.plot_confusion(title=title_1C_KDE,
                                    true_labels=true_labels,
                                    matrix=matrix_1C_KDE,
                                    metrics=metrics_1C_KDE,
                                    fig_size=[6, 5])

    # confusion matrix for 2-class only, KDE: noise and seismic
    two_class_true_1C_KDE, two_class_pred_1C_KDE= utils.two_class_convert(total_true_1C_KDE, total_pred_1C_KDE)
    two_class_matrix_1C_KDE, two_class_metrics_1C_KDE = utils.calc_confusion_metrics(two_class_true_1C_KDE, two_class_pred_1C_KDE)
    title_2Class_1C_KDE = f"{MODEL_TAG} {DATA_TAG}, 2-class 1C, KDE"
    fig_2Class_1C_KDE = utils.plot_confusion(title=title_2Class_1C_KDE,
                                        true_labels=["NO", "SE"],
                                        matrix=two_class_matrix_1C_KDE,
                                        metrics=two_class_metrics_1C_KDE,
                                        fig_size=[6, 5],
                                        subAdjust=(0.34, 0.66, 0.2, 0.8))


    # 1C confusion matrix, Norm
    matrix_1C_Norm, metrics_1C_Norm = utils.calc_confusion_metrics(total_true_1C_Norm, total_pred_1C_Norm)

    title_1C_Norm = f"{MODEL_TAG} {DATA_TAG}, 1C, Norm"
    fig_1C_Norm = utils.plot_confusion(title=title_1C_Norm,
                                    true_labels=true_labels,
                                    matrix=matrix_1C_Norm,
                                    metrics=metrics_1C_Norm,
                                    fig_size=[6, 5])

    
    #===================================================================
    #                            Save results
    #===================================================================

    logger.info(f"Save results ...")

    # save figures
    fig_1C_KDE_name = f"{title_1C_KDE}, {time_str}.jpg"
    fig_1C_KDE.savefig(os.path.join(save_dir, fig_1C_KDE_name), dpi=300)

    fig_2Class_1C_KDE_name = f"{title_2Class_1C_KDE}, {time_str}.jpg"
    fig_2Class_1C_KDE.savefig(os.path.join(save_dir, fig_2Class_1C_KDE_name), dpi=300)

    fig_1C_Norm_name = f"{title_1C_Norm}, {time_str}.jpg"
    fig_1C_Norm.savefig(os.path.join(save_dir, fig_1C_Norm_name), dpi=300)

    # save metrics
    dataset.save_json_data(os.path.join(save_dir, f"{fig_1C_KDE_name.split('.')[0]}.json"), metrics_1C_KDE)
    dataset.save_json_data(os.path.join(save_dir, f"{fig_2Class_1C_KDE_name.split('.')[0]}.json"), two_class_metrics_1C_KDE)
    dataset.save_json_data(os.path.join(save_dir, f"{fig_1C_Norm_name.split('.')[0]}.json"), metrics_1C_Norm)

    # save mismatches
    name_mismatch_noise_1C_KDE = f"Miss noise n{len(mismatch_noise_1C_KDE)}, {title_1C_KDE}.json"
    dataset.save_json_data(os.path.join(save_dir, name_mismatch_noise_1C_KDE), mismatch_noise_1C_KDE)

    name_mismatch_seismic_1C_KDE = f"Miss seismic n{len(mismatch_seismic_1C_KDE)}, {title_1C_KDE}.json"
    dataset.save_json_data(os.path.join(save_dir, name_mismatch_seismic_1C_KDE), mismatch_seismic_1C_KDE)

    logger.info("Task completed.")


[INFO] load test dataset ...


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Extreme SSD/stream_stead/data_stead/STEAD_MCQUAKE_5000_unseen.json'